# Multi-Agent Financial Analysis with Groq and smolagents

This notebook shows how to build a multi-agent financial analysis system using smolagents with Groq as the LLM backend.

The system uses three agents working together:

- A **Market Analyst** agent that reads stock price history and computes technical indicators
- A **Risk Manager** agent that computes portfolio risk metrics (Sharpe ratio, VaR, CVaR)
- An **Orchestrator** agent that coordinates both and delivers a final investment summary

## What you will learn

- How to connect smolagents to Groq using `LiteLLMModel`
- How to define custom financial tools using the `@tool` decorator
- How to build a multi-agent hierarchy with the manager/subordinate pattern
- How to run a coordinated financial analysis query across multiple specialized agents

## Requirements

- A free Groq API key from https://console.groq.com
- Python 3.10 or higher

## Step 1: Install Dependencies

In [ ]:
!pip install smolagents litellm yfinance pandas matplotlib python-dotenv -q

## Step 2: Import Libraries and Configure Groq

Groq is not a built-in smolagents model class, but it integrates cleanly through `LiteLLMModel`.
LiteLLM supports Groq out of the box. You just prefix the model name with `groq/`.

Set your `GROQ_API_KEY` as an environment variable or paste it directly below.

In [ ]:
import os
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

from smolagents import CodeAgent, LiteLLMModel, tool


warnings.filterwarnings("ignore")

# Set your Groq API key
os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"  # Replace with your actual key

# Initialize the Groq model via LiteLLM
# groq/llama-3.3-70b-versatile is Groq's recommended general-purpose model
model = LiteLLMModel(model_id="groq/llama-3.3-70b-versatile", temperature=0.1, max_tokens=4096)

print("Model ready:", model.model_id)

## Step 3: Define Custom Financial Tools

Each tool is a plain Python function decorated with `@tool`.
smolagents reads the docstring and type hints to understand what each tool does and how to call it.
Keep docstrings precise. Agents rely on them to decide when and how to use the tool.

In [ ]:
@tool
def get_stock_data(ticker: str, period: str = "3mo") -> str:
    """Fetches historical price data for a stock and returns key summary statistics.

    Args:
        ticker: Stock ticker symbol. Examples: 'AAPL', 'NVDA', 'MSFT', 'TSLA'.
        period: Time period for data. Options: '1mo', '3mo', '6mo', '1y', '2y'. Defaults to '3mo'.

    Returns:
        A string containing price summary statistics: start price, end price,
        total return, max, min, average daily volume, and annualized volatility.
    """
    stock = yf.Ticker(ticker)
    hist = stock.history(period=period)

    if hist.empty:
        return f"No data found for ticker '{ticker}'. Check the symbol and try again."

    start_price = round(hist["Close"].iloc[0], 2)
    end_price = round(hist["Close"].iloc[-1], 2)
    total_return = round(((end_price / start_price) - 1) * 100, 2)
    annualized_vol = round(hist["Close"].pct_change().std() * (252**0.5) * 100, 2)

    result = {
        "ticker": ticker.upper(),
        "period": period,
        "start_date": str(hist.index[0].date()),
        "end_date": str(hist.index[-1].date()),
        "start_price_usd": start_price,
        "end_price_usd": end_price,
        "total_return_pct": total_return,
        "max_price_usd": round(hist["Close"].max(), 2),
        "min_price_usd": round(hist["Close"].min(), 2),
        "avg_daily_volume": int(hist["Volume"].mean()),
        "annualized_volatility_pct": annualized_vol,
    }
    return str(result)

In [ ]:
@tool
def calculate_technical_indicators(ticker: str) -> str:
    """Calculates key technical indicators for a stock: RSI, MACD, and moving averages.

    Uses 6 months of daily price history for calculations.

    Args:
        ticker: Stock ticker symbol. Examples: 'AAPL', 'NVDA', 'MSFT'.

    Returns:
        A string containing RSI (14-day), MACD histogram value and signal,
        50-day and 200-day moving averages, and the price position relative to each MA.
    """
    stock = yf.Ticker(ticker)
    hist = stock.history(period="6mo")

    if hist.empty:
        return f"No data found for ticker '{ticker}'. Check the symbol and try again."

    close = hist["Close"]

    # RSI (14-day)
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    current_rsi = round(rsi.iloc[-1], 2)

    if current_rsi > 70:
        rsi_signal = "overbought"
    elif current_rsi < 30:
        rsi_signal = "oversold"
    else:
        rsi_signal = "neutral"

    # MACD (12/26/9)
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    macd_histogram = macd_line - signal_line
    macd_signal = "bullish" if macd_histogram.iloc[-1] > 0 else "bearish"

    # Moving Averages
    current_price = round(close.iloc[-1], 2)
    ma50 = round(close.rolling(50).mean().iloc[-1], 2)
    ma200 = round(close.rolling(200).mean().iloc[-1], 2) if len(close) >= 200 else None

    result = {
        "ticker": ticker.upper(),
        "current_price_usd": current_price,
        "rsi_14": current_rsi,
        "rsi_signal": rsi_signal,
        "macd_histogram": round(macd_histogram.iloc[-1], 4),
        "macd_signal": macd_signal,
        "ma_50_usd": ma50,
        "ma_200_usd": ma200 if ma200 else "insufficient data (need 200 days)",
        "price_vs_ma50": "above" if current_price > ma50 else "below",
        "price_vs_ma200": (("above" if current_price > ma200 else "below") if ma200 else "insufficient data"),
    }
    return str(result)

In [ ]:
@tool
def calculate_portfolio_risk(tickers_csv: str, weights_csv: str, period: str = "1y") -> str:
    """Calculates portfolio-level risk metrics: annualized return, volatility, Sharpe ratio, VaR, and CVaR.

    Args:
        tickers_csv: Comma-separated stock ticker symbols. Example: 'AAPL,NVDA,MSFT'.
        weights_csv: Comma-separated portfolio weights as decimals that must sum to 1.0.
            Example: '0.4,0.35,0.25' means 40% AAPL, 35% NVDA, 25% MSFT.
        period: Historical data period for calculations. Options: '6mo', '1y', '2y'. Defaults to '1y'.

    Returns:
        A string containing portfolio risk metrics: annualized return, annualized volatility,
        Sharpe ratio, 1-day 95% VaR, and 1-day 95% CVaR (Expected Shortfall).
    """
    tickers = [t.strip().upper() for t in tickers_csv.split(",")]
    weights = [float(w.strip()) for w in weights_csv.split(",")]

    if len(tickers) != len(weights):
        return "Error: number of tickers and weights must match."

    if abs(sum(weights) - 1.0) > 0.01:
        return f"Error: weights must sum to 1.0, but they sum to {round(sum(weights), 4)}."

    price_data = {}
    for ticker in tickers:
        stock = yf.Ticker(ticker)
        hist = stock.history(period=period)
        if not hist.empty:
            price_data[ticker] = hist["Close"]
        else:
            return f"Error: no data found for ticker '{ticker}'."

    prices = pd.DataFrame(price_data).dropna()
    returns = prices.pct_change().dropna()

    weights_arr = np.array(weights)
    portfolio_returns = returns.dot(weights_arr)

    annual_return = round(portfolio_returns.mean() * 252 * 100, 2)
    annual_vol = round(portfolio_returns.std() * (252**0.5) * 100, 2)

    risk_free_rate = 0.05  # approximate 1-year US Treasury rate
    sharpe = round(
        (portfolio_returns.mean() * 252 - risk_free_rate) / (portfolio_returns.std() * (252**0.5)),
        3,
    )

    # Value at Risk and Conditional VaR at 95% confidence, 1-day horizon
    var_95 = round(np.percentile(portfolio_returns, 5) * 100, 3)
    cvar_95 = round(
        portfolio_returns[portfolio_returns <= np.percentile(portfolio_returns, 5)].mean() * 100,
        3,
    )

    allocation = dict(zip(tickers, [f"{w * 100:.0f}%" for w in weights]))

    if annual_vol > 30:
        risk_rating = "High"
    elif annual_vol > 20:
        risk_rating = "Medium"
    else:
        risk_rating = "Low"

    result = {
        "portfolio_allocation": allocation,
        "period": period,
        "annualized_return_pct": annual_return,
        "annualized_volatility_pct": annual_vol,
        "sharpe_ratio": sharpe,
        "var_95_1day_pct": var_95,
        "cvar_95_1day_pct": cvar_95,
        "risk_rating": risk_rating,
        "note": "Sharpe ratio uses 5% risk-free rate. VaR and CVaR are historical simulation.",
    }
    return str(result)

### Quick tool test

Before building the agents, verify the tools work correctly on their own.

In [ ]:
# Test each tool independently
print("--- Stock Data ---")
print(get_stock_data("AAPL", "1mo"))

print("\n--- Technical Indicators ---")
print(calculate_technical_indicators("NVDA"))

print("\n--- Portfolio Risk ---")
print(calculate_portfolio_risk("AAPL,NVDA,MSFT", "0.4,0.35,0.25", "1y"))

## Step 4: Build the Multi-Agent System

I use `CodeAgent` for all three agents. Unlike `ToolCallingAgent`, `CodeAgent` writes and executes
Python code to call tools rather than using JSON tool-call format. This approach:

- Works reliably across LLM providers including Groq
- Lets agents chain and combine tool results with full Python expressiveness
- Uses fewer steps to reach a result (smolagents benchmark: ~30% fewer steps vs. tool-calling)

The **orchestrator** receives the user query and delegates to the two sub-agents via `managed_agents`.
It can call each sub-agent by name, passing context and collecting results.

In [ ]:
# Market Analyst: reads price history and technical signals
market_analyst = CodeAgent(
    tools=[get_stock_data, calculate_technical_indicators],
    model=model,
    name="market_analyst",
    description=(
        "Analyzes individual stocks using price history and technical indicators. "
        "Call it with a stock ticker symbol (and optionally a time period) to get "
        "a full technical breakdown including return, volatility, RSI, MACD, and moving averages."
    ),
    max_steps=6,
)

# Risk Manager: computes portfolio-level risk metrics
risk_manager = CodeAgent(
    tools=[calculate_portfolio_risk],
    model=model,
    name="risk_manager",
    description=(
        "Computes portfolio risk metrics for a set of stocks and their weights. "
        "Call it with a comma-separated list of tickers and their decimal weights "
        "(e.g. tickers='AAPL,NVDA,MSFT', weights='0.4,0.35,0.25'). "
        "Returns Sharpe ratio, annualized volatility, Value at Risk, and CVaR."
    ),
    max_steps=4,
)

# Orchestrator: coordinates both sub-agents and synthesizes the final answer
orchestrator = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[market_analyst, risk_manager],
    additional_authorized_imports=["json", "re"],
    max_steps=12,
)

print("Agents ready.")
print(f"  market_analyst tools : {[t.name for t in market_analyst.tools.values()]}")
print(f"  risk_manager tools   : {[t.name for t in risk_manager.tools.values()]}")
print(f"  orchestrator manages : {[a.name for a in orchestrator.managed_agents]}")

## Step 5: Run the Portfolio Analysis

I ask the orchestrator a single, real-world question. It will:

1. Delegate stock analysis for each ticker to `market_analyst`
2. Delegate portfolio risk calculation to `risk_manager`
3. Synthesize all results into a structured investment summary

In [ ]:
query = """
I want to analyze a technology portfolio consisting of AAPL (40%), NVDA (35%), and MSFT (25%).

Please do the following:
1. Use the market_analyst to get price data and technical indicators for each stock individually.
2. Use the risk_manager to calculate the portfolio risk metrics for this allocation over 1 year.
3. Based on all the data, give me a clear summary covering:
   - Which stock has the strongest and weakest technical setup right now
   - What the portfolio risk profile looks like (Sharpe ratio, VaR, volatility)
   - Whether this portfolio is suitable for a moderate-risk investor
   - One specific recommendation for improving the portfolio's risk-adjusted return
"""

result = orchestrator.run(query)
print(result)

## Step 6: Visualize Price Performance

Plot the normalized 1-year price performance of the three portfolio stocks for a quick visual comparison.
I normalize each stock to a base of 100 so the relative returns are directly comparable.

In [ ]:
TICKERS = ["AAPL", "NVDA", "MSFT"]
WEIGHTS = [0.40, 0.35, 0.25]
COLORS = ["#2196F3", "#4CAF50", "#FF9800"]

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Top chart: normalized individual stock returns
ax1 = axes[0]
for ticker, color in zip(TICKERS, COLORS):
    stock = yf.Ticker(ticker)
    hist = stock.history(period="1y")
    normalized = (hist["Close"] / hist["Close"].iloc[0]) * 100
    ax1.plot(hist.index, normalized, color=color, linewidth=1.8, label=ticker)

ax1.axhline(y=100, color="gray", linestyle="--", linewidth=0.9, alpha=0.6)
ax1.set_title("1-Year Normalized Price Performance (Base = 100)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Normalized Price")
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.25)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(ax1.get_xticklabels(), rotation=20)

# Bottom chart: portfolio return vs. equal-weight benchmark
ax2 = axes[1]
all_data = {}
for ticker in TICKERS:
    stock = yf.Ticker(ticker)
    hist = stock.history(period="1y")
    all_data[ticker] = hist["Close"]

prices_df = pd.DataFrame(all_data).dropna()
daily_returns = prices_df.pct_change().dropna()

weighted_returns = daily_returns.dot(np.array(WEIGHTS))
equal_returns = daily_returns.dot(np.ones(len(TICKERS)) / len(TICKERS))

portfolio_curve = (1 + weighted_returns).cumprod() * 100
equal_curve = (1 + equal_returns).cumprod() * 100

ax2.plot(
    portfolio_curve.index,
    portfolio_curve.values,
    color="#9C27B0",
    linewidth=2.0,
    label="Weighted Portfolio (40/35/25)",
)
ax2.plot(
    equal_curve.index,
    equal_curve.values,
    color="#607D8B",
    linewidth=1.5,
    linestyle="--",
    label="Equal-Weight Benchmark (33/33/33)",
)
ax2.axhline(y=100, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
ax2.set_title("Portfolio Cumulative Return vs. Equal-Weight Benchmark", fontsize=12, fontweight="bold")
ax2.set_ylabel("Cumulative Return (Base = 100)")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.25)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(ax2.get_xticklabels(), rotation=20)

plt.tight_layout(pad=2.0)
plt.savefig("portfolio_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved as portfolio_analysis.png")

## Summary

This notebook demonstrated how to build a coordinated multi-agent financial analysis pipeline using smolagents with Groq.

**Key patterns used:**

- `LiteLLMModel` with the `groq/` prefix connects smolagents to Groq with no extra setup
- `CodeAgent` writes Python to call tools, which works reliably across LLM providers
- The `managed_agents` parameter in `CodeAgent` creates a manager/subordinate hierarchy
- Custom `@tool` functions encapsulate domain logic cleanly with typed signatures and clear docstrings

**Extending this example:**

- Swap `groq/llama-3.3-70b-versatile` for any LiteLLM-supported model (e.g. `openai/gpt-4o`, `anthropic/claude-3-5-sonnet-20241022`)
- Add a fourth agent for news sentiment analysis using a web search tool
- Add more tools: dividend yield, P/E ratio, earnings surprise history
- Wrap the system in a Gradio UI using the pattern from `examples/gradio_ui.py`